# Navaja de Occam y parsimonia

**La idea:** si dos modelos explican los datos parecido, quedate con el más simple.

**Analogía:** se cortó la luz en tu casa. Hay dos explicaciones:

1. Saltó la térmica.

2. Un rayo dañó el transformador del barrio, justo mientras la empresa hacía un corte programado.

Las dos explican lo que pasó, pero la primera necesita menos cosas para ser cierta. Empezás por esa, y solo la complicás si aparece evidencia.

*Navaja de Occam (aplicación práctica del principio de parsimonia):* entre modelos que funcionan parecido, elegir el más simple, y complejizar solo si hay evidencia de que mejora. En árboles, esto se aplica limitando el crecimiento (maxdepth, minbucket, cp). Ojo: demasiado simple también falla (underfitting).

# sesgo en el muestreo

*La idea:* el modelo aprende de los datos que le das. Si los datos están sesgados, el modelo aprende ese mismo sesgo, por más bueno que sea el algoritmo.

Analogía: *texto en cursiva* querés medir la satisfacción con el centro de salud municipal y encuestás solo a la gente que va a la mañana. Si los que trabajan van a la tarde y tienen otra experiencia, tus conclusiones no representan a todos. Y no hay forma de arreglarlo con una cuenta mejor: el problema está en quién entró a la muestra.

*En ML pasa lo mismo:*

Un árbol entrenado solo con clientes de la ciudad va a predecir mal en los pueblos del interior.
Un modelo de créditos entrenado solo con gente a la que se le aprobó un crédito no sabe nada de los rechazados.

La clave: ningún hiperparámetro ni validación cruzada corrige esto. La CV divide los mismos datos sesgados, así que también da un resultado sesgado. Se resuelve en el diseño de la muestra.

# data snooping

*La idea:* el test nunca se usa durante el entrenamiento. Si lo mirás, aunque sea un poco, el resultado final te va a dar más optimista de lo que en realidad es.

*Analogía:* un alumno consigue el examen final antes de tiempo y “solo lo mira para orientarse”. Después saca 9, pero ese 9 no mide lo que sabe, porque estudió para esas preguntas.

*Formas de hacerlo sin darte cuenta:*

Probar 20 profundidades distintas mirando el resultado en test y quedarte con la mejor.
Estandarizar o imputar usando el promedio de toda la base, incluido el test, antes de separar.
Elegir variables según cuánto se relacionan con el target en toda la base.

*Cómo se evita (ya lo tenés en tus apuntes):*

Separar train y test al principio.
Hacer todo lo demás (preprocesar, elegir hiperparámetros con CV) solo dentro del train.
Usar el test una sola vez, al final.

# tres formas de buscar hiperparámetros

## Estrategias de tuning de hiperparámetros

**Tuning:** buscar los valores de los hiperparámetros que dan el mejor modelo. Cada combinación se evalúa con validación cruzada en el train, y el test se usa una sola vez, al final.

**Ejemplo:** árbol con `tree_depth` entre 1 y 30 y `min_n` entre 2 y 40. Probar todo serían 1.170 combinaciones; cada estrategia decide cuáles probar.

| | Grid search | Random search | Optimización bayesiana |
|---|---|---|---|
| **Qué defino yo** | Una lista de valores por hiperparámetro | Rangos y cantidad de pruebas | Rangos y cantidad de pruebas |
| **Cómo elige qué probar** | Todos los cruces de la lista | Combinaciones al azar dentro de los rangos | Según los resultados anteriores: arma un "mapa estimado" (proceso gaussiano) y prueba donde es más prometedor |
| **¿Aprende de las pruebas previas?** | No | No | Sí |
| **A favor** | Simple, exhaustiva dentro de la lista | Con las mismas pruebas explora más valores distintos | Llega a buen resultado con menos pruebas |
| **En contra** | Solo prueba lo que yo puse; las pruebas se multiplican con cada hiperparámetro | Es azar: puede dejar zonas sin explorar | Más compleja y secuencial (cada prueba espera a la anterior) |
| **Conviene cuando** | Pocos hiperparámetros y valores | Muchos hiperparámetros | Cada prueba es cara (modelos grandes) |
| **En tidymodels** | `tune_grid()` con `grid_regular()` | `tune_grid()` con `grid_random()` | `tune_bayes()` |

**Ejemplo de la bayesiana:** si bajar la profundidad de 12 a 10 mejoró el AUC, la próxima prueba sigue bajando (por ejemplo, a 8). No elige al azar: aprende de lo anterior.

## Hiperparámetros del árbol en tidymodels

**Idea:** tidymodels (`parsnip`) usa nombres comunes para todos los motores; la diapo 41 los traduce a `rpart`.

| parsnip (tidymodels) | rpart | Qué controla | Por defecto | Tipo | Para un árbol más simple |
|---|---|---|---|---|---|
| `tree_depth` | `maxdepth` | Profundidad máxima (cuántas preguntas seguidas) | 30 | Permiso | **Bajarlo** |
| `min_n` | `minsplit` | Mínimo de casos para intentar cortar un nodo | 20 | Exigencia | **Subirlo** |
| `cost_complexity` | `cp` | Cuánto debe mejorar un corte para hacerse | 0,01 | Exigencia | **Subirlo** |

**Más riesgo de overfitting cuando:** sube `tree_depth`, baja `min_n` o `cost_complexity` se acerca a 0.

**Marcar qué tunear:** `tune()` significa "este valor no lo fijo yo, buscalo con el tuning".

    arbol <- decision_tree(tree_depth = tune(), min_n = tune(), cost_complexity = tune()) |>
      set_engine("rpart") |>
      set_mode("classification")

**Para recordar:** permiso (`tree_depth`) más alto da más libertad para crecer; exigencias (`min_n`, `cost_complexity`) más altas hacen más difícil cortar. Ojo: en `ranger` o XGBoost, `min_n` se traduce al mínimo por hoja.

## Elecciones de diseño de un árbol

**1. ¿Cómo se hace un corte?**
- **Criterio para puntuar el corte:** Gini, ganancia de información o Gain Ratio.
- **Tipo de split:**

| Tipo de split | Cómo corta `Outlook` (Sunny, Overcast, Rainy) | Típico de |
|---|---|---|
| Binario (dos ramas) | ¿Es Sunny? → sí / no; después puede volver a preguntar dentro del "no" | CART (`rpart`) |
| Multicategórico (una rama por valor) | Sunny / Overcast / Rainy de una vez | ID3 y C4.5 |

**2. ¿Cuándo se deja de cortar? (criterios de parada)**

| Criterio | Qué significa | Ejemplo (golf) | Tipo |
|---|---|---|---|
| Nodo puro | Todos los casos son de la misma clase | Sunny + Windy = FALSE: 3 Yes | Natural |
| Sin variables útiles | Las variables que quedan tienen el mismo valor en todos los casos | Volver a preguntar `Outlook` dentro de Sunny | Natural |
| Límites que pone uno | Profundidad máxima, mínimo de casos para cortar o por hoja | `tree_depth`, `min_n` | Artificial: frena para evitar overfitting |

## Algoritmos clásicos de árboles: ID3, C4.5 y CART

Los tres construyen el árbol con el mismo flujo (ensayar cortes, puntuar, elegir y repetir); cambian las elecciones de diseño.

| | ID3 (Quinlan, 1986) | C4.5 (Quinlan, 1993) | CART (Breiman, 1984) |
|---|---|---|---|
| Qué es | El primero | Versión mejorada de ID3 | El más usado |
| Criterio de corte | Ganancia de información | Gain Ratio | Gini (clasificación) y varianza (regresión) |
| Tipo de split | Multicategórico | Multicategórico; binario en numéricas | Siempre binario |
| Variables numéricas | No | Sí (busca un punto de corte) | Sí |
| Faltantes | No | Sí (los ignora al calcular) | Según la implementación |
| Regresión | No | No | Sí |
| Contra el overfitting | Nada | Poda | Poda por costo-complejidad (`cp`) |
| Forma del árbol | Bajo y ancho | Bajo y ancho | Alto y angosto |

**Ejemplo (golf):** de `Outlook` salen 3 ramas de una vez, así que es estilo ID3 o C4.5. CART necesita dos preguntas: ¿Overcast? y, si no, ¿Sunny?

**Para recordar:** `rpart` es CART. Random Forest y XGBoost también usan árboles tipo CART.

# Gain Ratio

## Gain Ratio

**Problema que corrige:** la ganancia de información (ID3) favorece a las variables con muchos valores distintos. Ejemplo: un `ID del día` con 14 valores deja cada día en su propia hoja pura y tiene ganancia máxima (0,94), pero la regla no sirve para casos nuevos porque memoriza filas.

**Solución de C4.5:**

$$Gain\ Ratio = \frac{\text{Ganancia de información}}{\text{Información del split}}$$

| Pedacito | Qué es |
|---|---|
| Ganancia de información | Cuánta mezcla sacó la pregunta |
| Información del split | Entropía de cómo se reparten los casos entre las ramas (sin mirar el target). Muchas ramas → valor alto |
| Gain Ratio | Ganancia descontando cuánto fragmentó los datos |

**Ejemplo (golf):**

| Variable | Ganancia | Información del split | Gain Ratio |
|---|---|---|---|
| ID del día | 0,94 | 3,81 | 0,25 |
| Outlook | 0,25 | 1,58 | 0,16 |
| Humidity | 0,15 | 1,00 | 0,15 |
| Windy | 0,05 | 0,99 | 0,05 |

**Para recordar:**
- La diapo dice que se divide por la entropía del nodo raíz; lo preciso es por la información del split de esa variable.
- C4.5 además solo compara variables con ganancia al menos promedio, y en la práctica las columnas tipo ID se sacan antes de modelar.
- Gini contra entropía (diapo 51): en la práctica dan resultados muy parecidos; Gini es más rápido de calcular.